In [38]:
import rasterio
import numpy as np
import pandas as pd
import os

from scipy.stats import kstest, ks_2samp
from scipy.stats import wilcoxon
from scipy.stats import ttest_rel
from itertools import combinations

In [39]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]

In [59]:
def extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping):

    """
    Extract pixel values from TIFF files for each buoy in loc_boyas and for each specified date.
    """
    results = []
    target_dates = sorted(set(target_dates))
    tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]
    #print(tif_files)

    for date in target_dates:
        date_to_find = date.replace('-', '')
        matching = [f for f in tif_files if date_to_find in f]
        if matching:
            tiff_file = os.path.join(folder_path, matching[0])  

            with rasterio.open(tiff_file) as dataset:
                print(f"Processing {tiff_file}")
                bands = dataset.read()
                print(bands.shape[0])
                for _, row in loc_boyas.iterrows():
                    buoy_id = row["CodPuntoControl"].replace('-', '').strip()
                    lat = int(round(row["LatitudEPSG32630"]))
                    lon = int(round(row["LongitudEPSG32630"]))

                    try:
                        row_idx, col_idx = dataset.index(lon, lat)
                        #print(row_idx, col_idx)

                        if grouping == "3x3":
                            offset = 1
                        elif grouping == "5x5":
                            offset = 2
                        elif grouping == "9x9":
                            offset = 4
                        else:
                            offset = 0

                        if offset > 0:
                            window = (
                                slice(max(row_idx - offset, 0), min(row_idx + offset + 1, dataset.height)),
                                slice(max(col_idx - offset, 0), min(col_idx + offset + 1, dataset.width))
                            )
                            reflectances = bands[:, window[0], window[1]]
                            values = np.median(reflectances, axis=(1, 2))
                        else:
                            values = bands[:, row_idx, col_idx]

                        if values.shape == (8,):
                            band_names = {f"Band_{i+1}": val for i, val in enumerate(values)}
                        elif values.shape == (4,):
                            band_names = {f"Band_{i+2}": np.int16(val) for i, val in enumerate(values)}
                        else:
                            band_names = {f"Band_{i+1}": val for i, val in enumerate(values)} 

                        results.append({
                            "Date": date,
                            "Buoy": buoy_id,
                            "Latitude": lat,
                            "Longitude": lon,
                            **band_names
                        })

                    except IndexError:
                        print(f"Skipping {buoy_id} on {date}: Coordinates out of raster bounds")

                    #results = pd.DataFrame(results)[['Date', 'Buoy', 'Latitude', 'Longitude', 'Band_1', 'Band_2', 'Band_3','Band_4', 'Band_5', 'Band_6', 'Band_7', 'Band_8']]
    results = pd.DataFrame(results)

    return results

In [60]:

folder_path = "Copernicus/planet/"
target_dates = [
    '2018-02-20'
]

groupings = ["3x3"]#, "3x3", "5x5", "9x9"]
for grouping in groupings:

    df_tiffs = extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping)
    df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])

    df_tiffs.to_csv(f"saved_files/df_tifs_planet_{grouping}.csv", index=False)


Processing Copernicus/planet/20180220_composite.tif
4


In [62]:
df_planet = df_tiffs.copy()

In [63]:
df_tiffs

,Date,Buoy,Latitude,Longitude,Band_2,Band_3,Band_4,Band_5
0,2018-02-20,CTD1,4187246,695025,242,420,422,395
1,2018-02-20,CTD2,4181518,693105,233,421,399,429
2,2018-02-20,CTD3,4181698,695238,125,356,343,345
3,2018-02-20,CTD4,4180266,698264,97,294,284,311
4,2018-02-20,CTD5,4179450,700268,127,367,220,275
5,2018-02-20,CTD6,4176009,695829,65,275,287,329
6,2018-02-20,CTD7,4176724,690397,278,496,458,501
7,2018-02-20,CTD8,4174178,693048,245,452,397,361
8,2018-02-20,CTD9,4171106,693183,342,613,605,367
9,2018-02-20,CTD10,4170388,695646,197,422,385,329


,Date,Buoy,Latitude,Longitude,Band_2,Band_3,Band_4,Band_5
0,2018-02-20,CTD1,4187246,695025,2153,1972,1771,1775
1,2018-02-20,CTD2,4181518,693105,2109,1865,1594,1560
2,2018-02-20,CTD3,4181698,695238,2007,1768,1412,1349
3,2018-02-20,CTD4,4180266,698264,1902,1663,1328,1260
4,2018-02-20,CTD5,4179450,700268,1968,1721,1278,1216
5,2018-02-20,CTD6,4176009,695829,1904,1688,1320,1260
6,2018-02-20,CTD7,4176724,690397,1952,1745,1395,1348
7,2018-02-20,CTD8,4174178,693048,1937,1727,1365,1310
8,2018-02-20,CTD9,4171106,693183,2017,1878,1489,1387
9,2018-02-20,CTD10,4170388,695646,1894,1698,1329,1274


In [44]:
folder_path = "Copernicus/SAFE_downloads/subset_aoi/"
target_dates = [
    '2018-02-20'
]


groupings = ["1x1"]#, "3x3", "5x5", "9x9"]
for grouping in groupings:

    df_tiffs = extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping)
    df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])

    #df_tiffs.to_csv(f"saved_files/df_tifs_planet_{grouping}.csv", index=False)

Processing Copernicus/SAFE_downloads/subset_aoi/S2A_MSIL1C_20180220T105051_N0500_R051_T30SXG_20230910T163638_aoi_subset_10m.tif
8


In [45]:
df_sentinel = df_tiffs[['Date', 'Buoy', 'Latitude', 'Longitude','Band_2', 'Band_3','Band_4', 'Band_5']].copy()

### Tests estadísticos

In [49]:
df_sentinel.loc[:,"Band_2"]

0     2153
1     2109
2     2007
3     1902
4     1968
5     1904
6     1952
7     1937
8     2017
9     1894
10    1884
11    1903
Name: Band_2, dtype: uint16

In [52]:
reference = 'norm'  # Reference distribution (normal distribution)
bands = ['Band_2', 'Band_3','Band_4', 'Band_5']
groups = [df_planet, df_sentinel]
for group in groups:
    for band in bands:
        print(f"Evaluating {band} in {group}")
        # One Sample K-S Test
        stat, p_value = kstest(group.loc[:,band], reference)
        #print(f'K-S Test Statistic (One-Sample): {stat}')
        #print(f'P-value: {p_value}')
        alpha = 0.1
        if p_value > alpha:
            print("Fail to reject the null hypothesis: The sample comes from the specified distribution.")
        else:
            print("Reject the null hypothesis: The sample does not come from the specified distribution.")
        print()

Evaluating Band_2 in          Date   Buoy  Latitude  Longitude  Band_2  Band_3  Band_4  Band_5
0  2018-02-20   CTD1   4187246     695025   242.0   420.0   422.0   395.0
1  2018-02-20   CTD2   4181518     693105   233.0   421.0   399.0   429.0
2  2018-02-20   CTD3   4181698     695238   125.0   356.0   343.0   345.0
3  2018-02-20   CTD4   4180266     698264    97.0   294.0   284.0   311.0
4  2018-02-20   CTD5   4179450     700268   127.0   367.0   220.0   275.0
5  2018-02-20   CTD6   4176009     695829    65.0   275.0   287.0   329.0
6  2018-02-20   CTD7   4176724     690397   278.0   496.0   458.0   501.0
7  2018-02-20   CTD8   4174178     693048   245.0   452.0   397.0   361.0
8  2018-02-20   CTD9   4171106     693183   342.0   613.0   605.0   367.0
9  2018-02-20  CTD10   4170388     695646   197.0   422.0   385.0   329.0
10 2018-02-20  CTD11   4169609     700351   214.0   416.0   376.0   319.0
11 2018-02-20  CTD12   4173438     695414   101.0   350.0   320.0   296.0
Reject the null h

No siguen una distribución normal

In [53]:
bands = ['Band_2', 'Band_3','Band_4', 'Band_5']

for band in bands:
    print(f"Evaluating {band}")
    stat, p_value = wilcoxon(df_planet.loc[:,band], df_sentinel.loc[:,band])
    #print(f'Wilcoxon Signed-Rank Test Statistic: {stat}')
    #print(f'P-value: {p_value}')

    alpha = 0.05
    if p_value > alpha:
        print("Fail to reject the null hypothesis: The median difference is zero.")
    else:
        print("Reject the null hypothesis: The median difference is not zero.")

    # diffs = dfs[f"df_tiffs_c2x-complex-nets_{group}"][band] - dfs[f"df_tiffs_c2x-nets_{group}"][band]
    # effect_size = diffs.mean() / diffs.std()
    # print(f"Tamaño del efecto: {effect_size:.3f}")


print("----"*16)

Evaluating Band_2
Reject the null hypothesis: The median difference is not zero.
Evaluating Band_3
Reject the null hypothesis: The median difference is not zero.
Evaluating Band_4
Reject the null hypothesis: The median difference is not zero.
Evaluating Band_5
Reject the null hypothesis: The median difference is not zero.
----------------------------------------------------------------


**Tests de kolmogorov-smirnov de 2 samples** para ver si siguen la misma distribución

In [56]:
bands = ['Band_2', 'Band_3','Band_4', 'Band_5']

for band in bands:
    print(f"Evaluating {band}")
    #Two-Sample K-S Test
    stat, p_value = ks_2samp(df_planet.loc[:,band], df_sentinel.loc[:,band])


    alpha = 0.05
    if p_value > alpha:
        print("Fail to reject the null hypothesis: The two samples come from the same distribution.")
    else:
        print("Reject the null hypothesis: The two samples come from different distributions.")

Evaluating Band_2
Reject the null hypothesis: The two samples come from different distributions.
Evaluating Band_3
Reject the null hypothesis: The two samples come from different distributions.
Evaluating Band_4
Reject the null hypothesis: The two samples come from different distributions.
Evaluating Band_5
Reject the null hypothesis: The two samples come from different distributions.


**Mann-Whitney U test (muestras independientes)** para ver si hay una diferencia significativa entre los dos grupos

In [58]:

from scipy.stats import mannwhitneyu

bands = ['Band_2', 'Band_3','Band_4', 'Band_5']

for band in bands:
    print(f"Evaluating {band}")
    
    correlation = df_planet.loc[:,band].corr(df_sentinel.loc[:,band])

    print(f"Correlation for {band}: {correlation}")

Evaluating Band_2
Correlation for Band_2: 0.42602702053060076
Evaluating Band_3
Correlation for Band_3: 0.5253868027301857
Evaluating Band_4
Correlation for Band_4: 0.4967222741815476
Evaluating Band_5
Correlation for Band_5: 0.5351006828002406
